In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt
import warnings as ws
from sklearn.svm import SVC, LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import SequentialFeatureSelector
ws.filterwarnings('ignore')



def fim_feature_selection(indep_X,dep_Y,n):
    models = [
        ("RandomForestClassifier", RandomForestClassifier(n_estimators=10, criterion='entropy', random_state=0)),
           ]
    fwdlist = []

    for name, model in models:
       # SequentialFeatureSelector wraps the model and repeatedly tries adding
        # one feature at a time, keeping whichever addition improves the
        # cross-validated score the most, until `n` features are selected.
        sfs = SequentialFeatureSelector(
            model,
            n_features_to_select=n,
            direction='forward',
            scoring='accuracy',
            cv=5
        )

        # Fits the forward selection process using your features indep_X and target dep_Y.
        sfs.fit(indep_X, dep_Y)

        # get_support() returns a boolean mask of which columns were selected.
        selected_cols = indep_X.columns[sfs.get_support()].tolist()

        # Creates a new DataFrame with only the top selected features.
        selected_features = indep_X[selected_cols]
        fwdlist.append((name, selected_features))
    return fwdlist
    
#split_scalar - Split the input, output train and test set. then changes the input to scalar value    
def split_scalar(indep_X,dep_Y):
        X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test = sc.transform(X_test)    
        return X_train, X_test, y_train, y_test

# cm_prediction - used for classification method, model prediction evaluate confusion matrix and send acciracy, Report and send back 
def cm_prediction(classifier, X_test, y_test):
    y_pred = classifier.predict(X_test)

    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_test, y_pred)

    from sklearn.metrics import accuracy_score
    from sklearn.metrics import classification_report

    Accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    return classifier, Accuracy, report, X_test, y_test, cm

    
# logistic method is used for Linear regression model creation and r2 prediction
def logistic(X_train, y_train, X_test, y_test):
    from sklearn.linear_model import LogisticRegression
    classifier = LogisticRegression(random_state=0, max_iter=1000)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


def svm_linear(X_train, y_train, X_test, y_test):
    from sklearn.svm import SVC
    classifier = SVC(kernel='linear', random_state=0)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


def svm_NL(X_train, y_train, X_test, y_test):
    from sklearn.svm import SVC
    classifier = SVC(kernel='rbf', random_state=0)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


def Navie(X_train, y_train, X_test, y_test):
    from sklearn.naive_bayes import GaussianNB
    classifier = GaussianNB()
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


def knn(X_train, y_train, X_test, y_test):
    from sklearn.neighbors import KNeighborsClassifier
    classifier = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


def Decision(X_train, y_train, X_test, y_test):
    from sklearn.tree import DecisionTreeClassifier
    classifier = DecisionTreeClassifier(criterion='entropy', random_state=0)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


def random_forest(X_train, y_train, X_test, y_test):
    from sklearn.ensemble import RandomForestClassifier
    classifier = RandomForestClassifier(n_estimators=10, criterion='entropy', random_state=0)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


def gradient_boost(X_train, y_train, X_test, y_test):
    from sklearn.ensemble import GradientBoostingClassifier
    classifier = GradientBoostingClassifier(random_state=0)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)

    
# RFE_regression method is used for create dataset with columns name as 'Linear','SVMl','SVMnl','Decision','Random' and index ChiSquare
# and fill the each columns values 
def featureIm_Classification( accrf): 
    
     dataframe = pd.DataFrame(
        index=[ 'RandomForest'],
        columns=[ 'Random']
    )
    
     for number, idex in enumerate(dataframe.index):
        dataframe['Random'][idex] = accrf[number]
     return dataframe
    

In [2]:
# Read data from file and datatset should without index
dataset=pd.read_csv("prep.csv",index_col=None)
df2=dataset
# Preprocessed by one hot encoding
df2 = pd.get_dummies(df2, drop_first=True)
# assign the input only
indep_X=df2.drop('classification_yes', axis=1)
# assign output only
dep_Y=df2['classification_yes']

# choose the feature selection here using n feature 
featureImList=fim_feature_selection(indep_X,dep_Y,8)      


In [15]:
# Create 5 empty list for each algorithm and split the input and output
# Evalute each algorithmwise r2 score and send RFE_regression funtion
# finally the evalution data represent by table view.

for i, (name, X_selected) in enumerate(featureImList):
    print(name, X_selected.shape)
    X_train, X_test, y_train, y_test = split_scalar(X_selected, dep_Y)
    _, Accuracy, report, _, _, cm = random_forest(X_train, y_train, X_test, y_test)
    result = featureIm_Classification([Accuracy])
    print(result)

RandomForestClassifier (399, 8)
             Random
RandomForest   0.99


In [16]:
result
# 8

,Random
RandomForest,0.99
